[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C61_Detection_Practice_Interview_Course/01_experiment/01_experiment_design.ipynb)

# 01 · 实验设计与归因（种子方差 / 配对检验 / 功效分析 / 消融偏差 / 记录 schema）

目标：把「**+0.3 到底算不算提升**」这个问题，从一场辩论变成一次计算。

本 notebook 你会亲手实现：
1. **种子方差的生成模型**（$\mathrm{mAP}=\mu+s_j+\varepsilon$），并算出 5 个种子的期望极差
2. **单次对比的假阳性率**：真实差异为 0 时，看到「+0.3」的概率是多少（闭式 + 蒙特卡洛互相验证）
3. **配对设计**：同一批数据，配对分析 $p=0.002$、非配对分析 $p=0.20$ —— 设计决定你能不能看见效应
4. **Student-t 尾概率**（不用 scipy，自己写不完全 Beta 函数的连分数展开），并对已知临界值校验
5. **两种 bootstrap**：对种子重采样 vs 对评测图像重采样，它们回答不同的问题
6. **多重比较**：20 个改动 → 64% 概率至少一个假阳性；Bonferroni 与 Benjamini–Hochberg
7. **功效分析**：闭式公式 + 蒙特卡洛搜索，证明**正态近似在小样本下会低估所需种子数**
8. **消融的三种归因**：加法式 / 减法式 / Shapley —— 同一组实验，A 的贡献相差 10 倍
9. **调参预算的期望最优曲线**，以及「baseline 没调参」到底能偷走多少个点
10. **实验记录 schema 与公平性审计器**

> 心智模型：**没有方差估计的"提升"不是结论，是观测。
> 而把观测变成结论，只需要三样东西：配对设计、多种子、效应量与置信区间。**

## 1 · 种子方差的生成模型

$$\mathrm{mAP}_{ij} = \mu_i + s_j + \varepsilon_{ij}, \qquad
s_j\sim\mathcal N(0,\sigma_s^2)\ (\text{种子共享}),\quad
\varepsilon_{ij}\sim\mathcal N(0,\sigma_e^2)\ (\text{配置内独立})$$

取 $\sigma_s=0.22$、$\sigma_e=0.12$，于是单次 run 的标准差 $\sigma=0.2506$ ——
落在「检测任务典型 ±0.2–0.5」这个经验区间里。

In [ ]:
import math, json, itertools
import numpy as np
from statistics import NormalDist

SIGMA_S, SIGMA_E = 0.22, 0.12          # 种子共享 / 配置内独立
SIGMA = math.hypot(SIGMA_S, SIGMA_E)   # 单次 run 的总标准差
print(f'sigma_s={SIGMA_S}  sigma_e={SIGMA_E}  ->  单次 run 的 sigma = {SIGMA:.4f}')

def simulate_runs(mu, n_seeds, rng, seed_effects=None):
    '''模拟同一配置在 n_seeds 个种子下的 mAP。
       seed_effects 传入时表示"复用同一组种子"（= 配对设计）。'''
    s = rng.normal(0, SIGMA_S, size=n_seeds) if seed_effects is None else np.asarray(seed_effects)
    return mu + s + rng.normal(0, SIGMA_E, size=n_seeds), s

rng = np.random.default_rng(20260817)
runs, _ = simulate_runs(41.80, 5, rng)
print('\n同一份配置、同一份数据，只换种子跑 5 次：')
print('  mAP =', np.round(runs, 3))
print(f'  均值 {runs.mean():.3f}   标准差 {runs.std(ddof=1):.3f}   '
      f'极差 {runs.max()-runs.min():.3f}')

# 5 个种子的**期望极差**：标准正态下是 2.326 sigma
r2 = np.random.default_rng(0)
z = r2.normal(size=(100000, 5))
exp_range_std = float((z.max(1) - z.min(1)).mean())
print(f'\nE[极差 | n=5 标准正态] = {exp_range_std:.4f} sigma')
print(f'换算到 mAP：{exp_range_std * SIGMA:.4f} 点')
assert abs(exp_range_std - 2.326) < 0.02, exp_range_std
assert abs(exp_range_std * SIGMA - 0.583) < 0.02
print('\n⚠️  **跑 5 个种子，最好的一次比最差的一次高约 0.58 个 mAP —— 而它们完全相同。**')
print('    如果你只跑一次就和 baseline 比，你比较的一半是配置，一半是运气。')

## 2 · 「+0.3 是提升吗」：单次对比的假阳性率

真实差异 $\delta=0$ 时，观测到 $\Delta \ge 0.3$ 的概率是多少？
非配对时 $\sigma_\Delta=\sqrt2\,\sigma$，配对时 $\sigma_\Delta=\sqrt2\,\sigma_e$。

In [ ]:
ND = NormalDist()

def p_false_positive(threshold, sd_diff):
    '''真实差异为 0 时，观测差值 >= threshold 的概率。'''
    return 1.0 - ND.cdf(threshold / sd_diff)

SD_UNPAIRED = math.sqrt(2) * SIGMA        # 非配对：两边的种子效应都保留
SD_PAIRED   = math.sqrt(2) * SIGMA_E      # 配对：种子效应被抵消
print(f'非配对单次比较 sigma_delta = {SD_UNPAIRED:.4f}')
print(f'配对  单次比较 sigma_delta = {SD_PAIRED:.4f}   （砍掉了 {1-SD_PAIRED/SD_UNPAIRED:.0%}）\n')

print(f"{'观测到的"提升"':>14s} {'非配对假阳性率':>16s} {'配对假阳性率':>14s}")
for thr in [0.1, 0.2, 0.3, 0.5, 0.8, 1.0]:
    print(f'{thr:>14.1f} {p_false_positive(thr, SD_UNPAIRED):>16.1%} '
          f'{p_false_positive(thr, SD_PAIRED):>14.1%}')

p03 = p_false_positive(0.3, SD_UNPAIRED)
p05 = p_false_positive(0.5, SD_UNPAIRED)
assert abs(p03 - 0.1986) < 1e-3, p03
assert abs(p05 - 0.0791) < 1e-3, p05
assert abs(p_false_positive(0.3, SD_PAIRED) - 0.0385) < 1e-3

# 蒙特卡洛互相验证（闭式与模拟必须对得上，否则是公式写错了）
r3 = np.random.default_rng(7)
sim = r3.normal(0.0, SD_UNPAIRED, size=400000)
mc03 = float((sim >= 0.3).mean())
print(f'\n蒙特卡洛校验：P(>=0.3) 闭式 {p03:.4f} vs 模拟 {mc03:.4f}')
assert abs(mc03 - p03) < 0.005, (mc03, p03)
print('\n⚠️  **真实差异为 0 时，单次非配对对比有 19.9% 的概率显示"+0.3 的提升"。**')
print('    五次这样的对比里就有一次会骗到你。而一个季度做 20 次对比是很正常的。')
print('✅ 面试答法：先反问口径（哪个指标/几个种子/配对没有），再给这个 20% 的数。')

## 3 · 配对设计：免费的方差削减

配对不需要额外算力——同样跑 $2n$ 次训练，只是把种子对齐了。
下面用**同一批数据**做两种分析，看结论差多少。

In [ ]:
# 一次真实规模的实验：baseline 41.80，新方法真实高 0.35，5 个种子**配对**
n = 5
rg = np.random.default_rng(2)
s_shared = rg.normal(0, SIGMA_S, size=n)              # 两个配置共享的种子效应
A = 41.80 + s_shared + rg.normal(0, SIGMA_E, size=n)  # baseline
B = 41.80 + 0.35 + s_shared + rg.normal(0, SIGMA_E, size=n)   # 新方法

print(f"{'种子':>4s} {'baseline':>10s} {'新方法':>10s} {'配对差':>8s}")
for j in range(n):
    print(f'{j:>4d} {A[j]:>10.3f} {B[j]:>10.3f} {B[j]-A[j]:>+8.3f}')
d = B - A
print(f'\n均值      {A.mean():>10.3f} {B.mean():>10.3f} {d.mean():>+8.3f}')
print(f'标准差    {A.std(ddof=1):>10.3f} {B.std(ddof=1):>10.3f} {d.std(ddof=1):>8.3f}')

# 关键观察：**单边标准差 0.32/0.40，而配对差的标准差只有 0.105**
assert A.std(ddof=1) > 0.25 and B.std(ddof=1) > 0.25
assert d.std(ddof=1) < 0.15
corr = float(np.corrcoef(A, B)[0, 1])
print(f'\nA 与 B 的配对相关系数 r = {corr:.3f}  （理论值 sigma_s^2/(sigma_s^2+sigma_e^2) = '
      f'{SIGMA_S**2/(SIGMA_S**2+SIGMA_E**2):.3f}）')
assert corr > 0.9
print('\n✅ 配对差的标准差 0.105 << 单边标准差 0.32 —— 种子效应在做差时被抵消了。')
print('   **这就是"免费的方差削减"：同样 10 次训练，配对设计的信噪比高一倍以上。**')

In [ ]:
# 配对是否真的起作用？做一个可以直接用在项目里的诊断
def pairing_diagnostic(a, b):
    '''若配对无效（相关性≈0），s_d 会接近 sqrt(2)*s_单边；
       若配对有效，s_d 会显著小于它。返回 (s_d, 无效时的期望 s_d, 方差削减比)。'''
    a, b = np.asarray(a, float), np.asarray(b, float)
    s_side = math.sqrt(0.5 * (a.var(ddof=1) + b.var(ddof=1)))
    s_d = (b - a).std(ddof=1)
    return s_d, math.sqrt(2) * s_side, 1 - s_d / (math.sqrt(2) * s_side)

s_d, s_d_if_useless, cut = pairing_diagnostic(A, B)
print(f'实际配对差标准差       s_d = {s_d:.4f}')
print(f'若配对完全无效应为     s_d = {s_d_if_useless:.4f}')
print(f'方差削减              {cut:.1%}')
assert cut > 0.5, '本例中配对应削减一半以上'

# 反例：改动改变了随机性本身的作用（如换优化器），种子效应不再共享
# 单次 n=5 的削减比噪声很大，所以做 2000 次复现取均值
rg2 = np.random.default_rng(5)
cuts_shared, cuts_broken = [], []
for _ in range(2000):
    s0 = rg2.normal(0, SIGMA_S, size=n)
    a1 = 41.80 + s0 + rg2.normal(0, SIGMA_E, size=n)
    b1 = 42.15 + s0 + rg2.normal(0, SIGMA_E, size=n)          # 种子效应共享
    cuts_shared.append(pairing_diagnostic(a1, b1)[2])
    a2 = 41.80 + rg2.normal(0, SIGMA_S, size=n) + rg2.normal(0, SIGMA_E, size=n)
    b2 = 42.15 + rg2.normal(0, SIGMA_S, size=n) + rg2.normal(0, SIGMA_E, size=n)  # 不共享
    cuts_broken.append(pairing_diagnostic(a2, b2)[2])
c_ok, c_bad = float(np.mean(cuts_shared)), float(np.mean(cuts_broken))
print(f'\n2000 次复现的平均方差削减：')
print(f'  种子效应共享（配对有效）  {c_ok:.1%}   （理论上限 1 - sigma_e/sigma = '
      f'{1 - SIGMA_E/SIGMA:.1%}）')
print(f'  种子效应不共享（配对失效）{c_bad:.1%}')
assert c_ok > 0.40 and c_bad < 0.10, (c_ok, c_bad)
print('⚠️  **配对不是万能的**：如果改动改变了随机性的作用（换优化器/换整套增强/换 backbone），')
print('    同种子下两边的相关性会掉下来，配对收益变小 —— 这时只能老实加种子。')
print('✅ 所以每次配对实验都应该跑一遍这个诊断，而不是假设配对一定有效。')

## 4 · 配对 t 检验：自己实现 Student-t 的尾概率

不用 scipy。双侧 $p$ 值可以写成正则化不完全 Beta 函数：
$$p = I_{\nu/(\nu+t^2)}\!\left(\tfrac{\nu}{2},\ \tfrac12\right)$$
用 Lentz 连分数算 $I_x(a,b)$，几十行标准库就够。

In [ ]:
def _betacf(a, b, x, itmax=300, eps=3e-16):
    '''不完全 Beta 函数的连分数展开（Lentz 修正算法）。'''
    qab, qap, qam = a + b, a + 1.0, a - 1.0
    c = 1.0
    d = 1.0 - qab * x / qap
    if abs(d) < 1e-300: d = 1e-300
    d = 1.0 / d
    h = d
    for m in range(1, itmax + 1):
        m2 = 2 * m
        aa = m * (b - m) * x / ((qam + m2) * (a + m2))
        d = 1.0 + aa * d
        if abs(d) < 1e-300: d = 1e-300
        c = 1.0 + aa / c
        if abs(c) < 1e-300: c = 1e-300
        d = 1.0 / d
        h *= d * c
        aa = -(a + m) * (qab + m) * x / ((a + m2) * (qap + m2))
        d = 1.0 + aa * d
        if abs(d) < 1e-300: d = 1e-300
        c = 1.0 + aa / c
        if abs(c) < 1e-300: c = 1e-300
        d = 1.0 / d
        delta = d * c
        h *= delta
        if abs(delta - 1.0) < eps:
            break
    return h

def betai(a, b, x):
    '''正则化不完全 Beta 函数 I_x(a, b)。'''
    if x <= 0: return 0.0
    if x >= 1: return 1.0
    lbeta = math.lgamma(a + b) - math.lgamma(a) - math.lgamma(b)
    bt = math.exp(lbeta + a * math.log(x) + b * math.log(1 - x))
    if x < (a + 1) / (a + b + 2):
        return bt * _betacf(a, b, x) / a
    return 1.0 - bt * _betacf(b, a, 1 - x) / b

def t_pvalue(t, df):
    '''双侧 p 值。'''
    t = abs(float(t))
    return betai(0.5 * df, 0.5, df / (df + t * t))

def t_crit(df, alpha=0.05):
    '''双侧临界值：二分求 t 使 t_pvalue(t, df) = alpha。'''
    lo, hi = 0.0, 1000.0
    for _ in range(200):
        mid = 0.5 * (lo + hi)
        if t_pvalue(mid, df) > alpha:
            lo = mid
        else:
            hi = mid
    return 0.5 * (lo + hi)

# —— 对照标准 t 表校验（这些临界值可以查表核对）——
print(f"{'df':>8s} {'t_0.975 (查表)':>16s} {'本实现':>12s} {'p(该 t)':>10s}")
for df, ref in [(2, 4.3027), (4, 2.7764), (5, 2.5706), (9, 2.2622), (20, 2.0860), (100, 1.9840)]:
    got = t_crit(df)
    print(f'{df:>8d} {ref:>16.4f} {got:>12.4f} {t_pvalue(ref, df):>10.5f}')
    assert abs(got - ref) < 2e-3, (df, got, ref)
    assert abs(t_pvalue(ref, df) - 0.05) < 1e-3
assert t_pvalue(0.0, 5) == 1.0
assert t_pvalue(1.0, 10) > t_pvalue(2.0, 10) > t_pvalue(3.0, 10)   # 单调
print('\n✅ t 分布尾概率实现正确（与标准 t 表在 2e-3 内一致）。')

In [ ]:
def paired_ttest(a, b, alpha=0.05):
    '''配对 t 检验。返回 (mean_diff, t, p, ci_lo, ci_hi)。'''
    a, b = np.asarray(a, float), np.asarray(b, float)
    d = b - a
    n = len(d)
    md, sd = float(d.mean()), float(d.std(ddof=1))
    se = sd / math.sqrt(n)
    t = md / se
    tc = t_crit(n - 1, alpha)
    return md, t, t_pvalue(t, n - 1), md - tc * se, md + tc * se

def unpaired_ttest(a, b, alpha=0.05):
    '''两独立样本 t 检验（等方差）。'''
    a, b = np.asarray(a, float), np.asarray(b, float)
    na, nb = len(a), len(b)
    sp = math.sqrt(((na-1)*a.var(ddof=1) + (nb-1)*b.var(ddof=1)) / (na + nb - 2))
    se = sp * math.sqrt(1/na + 1/nb)
    t = (b.mean() - a.mean()) / se
    return float(b.mean() - a.mean()), t, t_pvalue(t, na + nb - 2)

md_, t_, p_, lo_, hi_ = paired_ttest(A, B)
md_u, t_u, p_u = unpaired_ttest(A, B)
print('同一批数据（第 3 节那 5 对），两种分析方式：\n')
print(f'  【配对】  差值 {md_:+.4f}   t={t_:.3f} (df={n-1})   p={p_:.5f}   95% CI [{lo_:+.3f}, {hi_:+.3f}]')
print(f'  【非配对】差值 {md_u:+.4f}   t={t_u:.3f} (df={2*n-2})   p={p_u:.4f}')
assert p_ < 0.01 and p_u > 0.10, (p_, p_u)
assert lo_ > 0, '配对下 95% CI 不含 0 -> 可以下结论'
print('\n⚠️  **同一批数据、同一个真实效应（+0.35）：**')
print(f'    配对分析   p = {p_:.4f}  -> 显著，CI 不含 0，可以下结论')
print(f'    非配对分析 p = {p_u:.4f}  -> 不显著，"无结论"')
print('    差别**完全来自实验设计**，不是来自数据、模型或统计工具。')
print('\n✅ 报告规范：报 **效应量 + 置信区间**，p 值只是附带。')
print(f'   本例的规范写法：「+{md_:.2f} mAP（95% CI [{lo_:.2f}, {hi_:.2f}]，n=5 配对，p={p_:.3f}）」')
print(f'   效应量也可以用 sigma 为单位：+{md_/SIGMA:.2f} sigma —— 跨项目可比。')

## 5 · 两种 bootstrap：它们回答不同的问题

- **对种子重采样** → 「如果我再跑一次训练，差值会落在哪」
- **对评测图像重采样** → 「如果换一批同分布的评测数据，指标会落在哪」

第二种**不会随着多跑种子而消失**——它是评测集本身的抽样不确定性。

In [ ]:
def bootstrap_ci(values, stat=np.mean, n_boot=20000, alpha=0.05, seed=0):
    '''百分位法 bootstrap 置信区间。'''
    v = np.asarray(values, float)
    r = np.random.default_rng(seed)
    idx = r.integers(0, len(v), size=(n_boot, len(v)))
    boots = stat(v[idx], axis=1)
    lo, hi = np.percentile(boots, [100*alpha/2, 100*(1-alpha/2)])
    return float(boots.mean()), float(lo), float(hi)

# ① 对**种子**重采样：配对差的分布
mb, lb, hb = bootstrap_ci(B - A, seed=1)
print(f'① 对种子重采样  配对差 {mb:+.3f}  95% CI [{lb:+.3f}, {hb:+.3f}]   (n=5)')
print(f'   对照配对 t 的 CI                    [{lo_:+.3f}, {hi_:+.3f}]')
assert lb > 0, '两种方法都应给出不含 0 的区间'
print('   ⚠️  n=5 时 bootstrap 的分位数很粗糙（只有 5 个不同取值可抽），')
print('       这种情形**宁可用配对 t**；bootstrap 的优势要 n>=20 才体现。\n')

# ② 对**评测图像**重采样：绝对指标的不确定度
#    合成 2000 张图的"逐图 AP 贡献"（Beta 分布：多数图好、少数图差 —— 贴近真实长尾）
rimg = np.random.default_rng(3)
per_image_ap = rimg.beta(6.0, 1.4, size=2000)
proxy_map = float(per_image_ap.mean())
mi, li, hi_ = bootstrap_ci(per_image_ap, n_boot=5000, seed=4)
print(f'② 对评测图像重采样（2000 张）  代理 mAP {proxy_map:.4f}  95% CI [{li:.4f}, {hi_:.4f}]')
print(f'   区间半宽 = {(hi_-li)/2*100:.3f} 个百分点')
assert abs(mi - proxy_map) < 0.005
half_2000 = (hi_ - li) / 2

# 评测集只有 1/4 大时呢？
small = per_image_ap[:500]
_, ls, hs = bootstrap_ci(small, n_boot=5000, seed=5)
half_500 = (hs - ls) / 2
print(f'   若评测集只有 500 张           95% CI 半宽 = {half_500*100:.3f} 个百分点')
print(f'   半宽之比 {half_500/half_2000:.2f}  ≈ sqrt(4) = 2  —— **不确定度按 1/sqrt(N) 缩小**')
assert 1.6 < half_500 / half_2000 < 2.5, half_500 / half_2000
print('\n⚠️  注意：真实 mAP 不是"逐图 AP 的平均"（它是全局排序后的面积），')
print('    这里用逐图贡献做**代理**以便 bootstrap。真实实现要按图像整体重采样后**重算 mAP**。')
print('    但结论不变：**评测集抽样噪声不随多跑种子而消失**，只能靠加数据。')
print('📌 TSR 落点：夜间桶只有几百个实例 -> 它的 CI 半宽是整体 mAP 的 2–3 倍。')
print('    所以分桶门禁的阈值必须按桶各自的 sigma 设，统一阈值一定会误报警。')

## 6 · 多重比较：试得越多，越容易「发现」不存在的东西

In [ ]:
m_tests = 20
alpha = 0.05
fwer = 1 - (1 - alpha) ** m_tests
print(f'一个季度做 {m_tests} 次独立对比，每次用 alpha={alpha}：')
print(f'  **至少出现一个假阳性的概率 = 1 - 0.95^20 = {fwer:.1%}**')
assert abs(fwer - 0.6415) < 1e-3

def bonferroni(pvals, alpha=0.05):
    thr = alpha / len(pvals)
    return sorted(i for i, p in enumerate(pvals) if p <= thr), thr

def benjamini_hochberg(pvals, q=0.05):
    '''控制 FDR：p 升序排，找最大的 k 使 p_(k) <= k/m*q，拒绝前 k 个。'''
    m = len(pvals)
    order = sorted(range(m), key=lambda i: pvals[i])
    k = 0
    for rank, i in enumerate(order, start=1):
        if pvals[i] <= rank / m * q:
            k = rank
    return sorted(order[:k])

# 一个季度的 10 个改动的 p 值（3 个真有效，7 个纯噪声）
P = [0.001, 0.008, 0.039, 0.041, 0.042, 0.060, 0.074, 0.205, 0.212, 0.216]
naive = [i for i, p in enumerate(P) if p <= 0.05]
bonf, thr = bonferroni(P)
bh = benjamini_hochberg(P, q=0.05)
print(f'\n{len(P)} 个改动的 p 值: {P}')
print(f'  朴素 p<0.05        -> 判定显著的有 {len(naive)} 个: {naive}')
print(f'  Bonferroni (a/m={thr:.4f}) -> {len(bonf)} 个: {bonf}   （控 FWER，很保守）')
print(f'  Benjamini-Hochberg -> {len(bh)} 个: {bh}   （控 FDR，探索期更实用）')
assert naive == [0, 1, 2, 3, 4]
assert bonf == [0]
assert bh == [0, 1], bh
print('\n✅ 同一组 p 值，三种口径给出 5 / 1 / 2 个"显著" —— **先说清用哪种口径**。')
print('   工程建议：探索阶段用 BH 生成"值得跟进的候选清单"；')
print('   上线门禁用 Bonferroni（宁可漏掉真提升，不能放进假提升）。')
print('   而比统计校正更好用的是**分阶段验证**：dev 上随便试，holdout 上每个改动只跑一次。')

## 7 · 功效分析：需要多少个种子

$$n \ge \frac{(z_{1-\alpha/2}+z_{1-\beta})^2\,\sigma_\Delta^2}{\delta^2}
= \frac{7.849\,\sigma_\Delta^2}{\delta^2},
\qquad
\delta_{\min} = 2.80\,\frac{\sigma_\Delta}{\sqrt n}$$

先算闭式，再用蒙特卡洛校验——**你会发现闭式在小样本下严重低估**。

In [ ]:
Z_SUM = ND.inv_cdf(0.975) + ND.inv_cdf(0.80)     # 1.96 + 0.8416
print(f'z_(1-a/2) + z_(1-b) = {Z_SUM:.4f}   平方 = {Z_SUM**2:.4f}')
assert abs(Z_SUM**2 - 7.8489) < 1e-3

def n_closed(delta, sd_diff, alpha=0.05, power=0.80):
    z = ND.inv_cdf(1 - alpha/2) + ND.inv_cdf(power)
    return math.ceil(z * z * sd_diff * sd_diff / (delta * delta))

def mde_closed(n, sd_diff, alpha=0.05, power=0.80):
    z = ND.inv_cdf(1 - alpha/2) + ND.inv_cdf(power)
    return z * sd_diff / math.sqrt(n)

print(f"\n{'要检出的效应':>12s} {'非配对 n/组':>12s} {'总训练次数':>11s} "
      f"{'配对 n 对':>10s} {'总训练次数':>11s}")
for delta in [0.1, 0.3, 0.5, 1.0]:
    nu = n_closed(delta, SD_UNPAIRED)
    npd = n_closed(delta, SD_PAIRED)
    print(f'{delta:>12.1f} {nu:>12d} {2*nu:>11d} {npd:>10d} {2*npd:>11d}')

assert n_closed(0.3, SD_UNPAIRED) == 11
assert n_closed(0.3, SD_PAIRED) == 3
assert n_closed(1.0, SD_UNPAIRED) == 1, '+1.0 这种大改动单次对比就够 —— 大改动不需要统计学'
assert n_closed(0.1, SD_UNPAIRED) == 99
print('\n✅ 闭式结论：检出 +0.3 需要非配对 22 次训练 / 配对 6 次；')
print('   检出 +0.1 需要 198 次 —— **这个量级的效应在检测里基本无法证实**。')

In [ ]:
# —— 蒙特卡洛校验：闭式用的是正态分位数，而小样本服从 t 分布（尾巴更厚）——
def sim_power_paired(n, delta, trials=4000, seed=1):
    r = np.random.default_rng(seed)
    d = r.normal(delta, SD_PAIRED, size=(trials, n))
    t = d.mean(1) / (d.std(1, ddof=1) / math.sqrt(n))
    return float((t > t_crit(n - 1)).mean())        # 双侧 0.05 且方向为正

def sim_power_unpaired(n, delta, trials=4000, seed=2):
    r = np.random.default_rng(seed)
    a = r.normal(0.0, SIGMA, size=(trials, n))
    b = r.normal(delta, SIGMA, size=(trials, n))
    sp = np.sqrt((a.var(1, ddof=1) + b.var(1, ddof=1)) / 2)
    t = (b.mean(1) - a.mean(1)) / (sp * math.sqrt(2.0 / n))
    return float((t > t_crit(2 * n - 2)).mean())

def search_n(power_fn, delta, target=0.80, n_max=40):
    for n in range(2, n_max + 1):
        if power_fn(n, delta) >= target:
            return n
    return None

print(f"{'配对 n':>8s} {'t 临界值':>10s} {'实际功效(delta=0.3)':>20s}")
for n_ in range(2, 9):
    print(f'{n_:>8d} {t_crit(n_-1):>10.3f} {sim_power_paired(n_, 0.3):>20.1%}')

n_sim_p = search_n(sim_power_paired, 0.3)
n_sim_u = search_n(sim_power_unpaired, 0.3)
print(f'\n达到 80% 功效所需（delta=0.3）：')
print(f'  配对   闭式 {n_closed(0.3, SD_PAIRED):>2d} 对   蒙特卡洛 {n_sim_p:>2d} 对')
print(f'  非配对 闭式 {n_closed(0.3, SD_UNPAIRED):>2d} /组  蒙特卡洛 {n_sim_u:>2d} /组')
assert n_closed(0.3, SD_PAIRED) < n_sim_p, '闭式（正态近似）在小样本下会低估所需重复数'
assert 4 <= n_sim_p <= 8, n_sim_p
assert n_sim_u >= 2 * n_sim_p, (n_sim_u, n_sim_p)
print('\n⚠️  n=3 时自由度只有 2，双侧 0.05 的 t 临界值是 **4.303** 而不是 1.96 ——')
print('    闭式给的 3 对，实际功效只有 40%。**小样本下所有正态近似的公式都要用模拟校验。**')

In [ ]:
# —— 反过来用：预算定死了，你到底能看见多大的效应（MDE）——
def sim_mde(n, target=0.80, grid=None):
    '''给定 n，搜出实际能以 target 功效检出的最小效应。'''
    grid = np.arange(0.05, 1.50, 0.01) if grid is None else grid
    for d in grid:
        if sim_power_paired(n, float(d)) >= target:
            return float(d)
    return None

print(f"{'预算(配对对数)':>14s} {'闭式 MDE':>10s} {'该 delta 下实际功效':>20s} {'真实 MDE':>10s}")
mdes = {}
for n_ in [3, 5, 10]:
    mc = mde_closed(n_, SD_PAIRED)
    mdes[n_] = sim_mde(n_)
    print(f'{n_:>14d} {mc:>10.3f} {sim_power_paired(n_, mc):>20.1%} {mdes[n_]:>10.3f}')

assert mdes[3] > 1.5 * mde_closed(3, SD_PAIRED), '3 对时闭式 MDE 乐观了一倍'
assert mdes[3] > mdes[5] > mdes[10], 'MDE 必须随预算单调下降'
print('\n⚠️  **只有 3 对预算时，你真正能可靠看见的最小效应是 0.56，不是闭式说的 0.27。**')
print('✅ 开跑之前先算 MDE：如果 MDE 比你期待的效应还大，这个实验不值得跑 ——')
print('   要么加预算，要么改设计（配对/共享评测集），要么换一个效应更大的改动。')
print('\n📌 一条必须分清的表述差别：')
print('   「没有证据说明它有用」 != 「有证据说明它没用」')
print('   MDE 之下的结果只能报**前者**。把前者说成后者，是实验解读里最常见的错误。')

## 8 · 消融设计的偏差：加法式 / 减法式 / Shapley

三个组件 A（Mosaic）、B（copy-paste）、C（更长训练），baseline 40.0。
真实响应含交互：A×B **冗余** −0.8（都在制造小目标）、B×C **协同** +0.5、
A×C −0.3、三阶 +0.2。**完整系统 42.2，总增益 +2.2。**

In [ ]:
BASE = 40.0
MAIN  = {'A': 1.0, 'B': 1.0, 'C': 0.6}
INTER = {('A', 'B'): -0.8, ('B', 'C'): 0.5, ('A', 'C'): -0.3, ('A', 'B', 'C'): 0.2}
COMPS = ['A', 'B', 'C']
NAMES = {'A': 'Mosaic', 'B': 'copy-paste', 'C': '更长训练'}

def perf(S):
    '''子集 S 的真实性能（我们这里"知道"真值；现实中每个 S 要跑一次训练）。'''
    S = frozenset(S)
    v = BASE + sum(MAIN[x] for x in S)
    for key, val in INTER.items():
        if set(key) <= S:
            v += val
    return v

full = perf(COMPS)
total_gain = full - perf([])
print(f'baseline = {perf([]):.2f}   完整系统 = {full:.2f}   总增益 = {total_gain:+.2f}\n')
print('全部 2^3 = 8 个子集（这就是"全子集消融"要跑的实验数）：')
for r in range(4):
    for S in itertools.combinations(COMPS, r):
        print(f"  {'{' + ','.join(S) + '}':<10s} {perf(S):.2f}")
assert abs(full - 42.2) < 1e-9 and abs(total_gain - 2.2) < 1e-9

In [ ]:
def additive(x):     return perf([x]) - perf([])                 # 从 baseline 加一个
def subtractive(x):  return perf(COMPS) - perf([c for c in COMPS if c != x])   # 从 full 去一个

def shapley(x):
    '''对所有加入顺序求平均的边际贡献。'''
    others = [c for c in COMPS if c != x]
    n = len(COMPS)
    tot = 0.0
    for r in range(len(others) + 1):
        for S in itertools.combinations(others, r):
            w = math.factorial(len(S)) * math.factorial(n - len(S) - 1) / math.factorial(n)
            tot += w * (perf(set(S) | {x}) - perf(S))
    return tot

add = {x: additive(x) for x in COMPS}
sub = {x: subtractive(x) for x in COMPS}
sha = {x: shapley(x) for x in COMPS}

print(f"{'组件':<16s} {'加法式':>9s} {'减法式':>9s} {'Shapley':>9s}   结论")
for x in COMPS:
    note = '**相差 10 倍**' if abs(add[x] - sub[x]) > 0.5 else ''
    print(f'{x + " " + NAMES[x]:<16s} {add[x]:>+9.2f} {sub[x]:>+9.2f} {sha[x]:>+9.2f}   {note}')
print(f'{"合计":<16s} {sum(add.values()):>+9.2f} {sum(sub.values()):>+9.2f} '
      f'{sum(sha.values()):>+9.2f}   真实总增益 {total_gain:+.2f}')

assert abs(add['A'] - 1.00) < 1e-9 and abs(sub['A'] - 0.10) < 1e-9
assert abs(sha['A'] - 0.5166666666) < 1e-6
# Shapley 的**效率公理**：各组件的分摊之和 == 总增益（另两种都不满足）
assert abs(sum(sha.values()) - total_gain) < 1e-9, sum(sha.values())
assert abs(sum(add.values()) - total_gain) > 0.3, '加法式合计 2.60 != 2.20'
assert abs(sum(sub.values()) - total_gain) > 0.15, '减法式合计 2.00 != 2.20'
print('\n⚠️  **A 的贡献：加法式说 +1.00，减法式说 +0.10 —— 相差 10 倍，而两者都没算错。**')
print('    加法式回答「只能加一个时加哪个」；减法式回答「要砍一个时砍哪个代价最小」。')
print('    把任何一个说成"A 的贡献"这个唯一数字，都是错的。')

In [ ]:
# 交互作用可以被直接测出来 —— 而它往往比主效应更有信息量
def interaction_2way(x, y):
    '''二阶交互 = f(xy) - f(x) - f(y) + f({})，正=协同，负=冗余。'''
    return perf([x, y]) - perf([x]) - perf([y]) + perf([])

print('实测二阶交互（从全子集实验里免费得到）：')
for x, y in itertools.combinations(COMPS, 2):
    v = interaction_2way(x, y)
    kind = '协同（1+1>2）' if v > 0.05 else ('冗余（做的是同一件事）' if v < -0.05 else '基本独立')
    print(f'  {NAMES[x]:<10s} x {NAMES[y]:<10s} {v:>+6.2f}   {kind}')
assert abs(interaction_2way('A', 'B') - (-0.8)) < 1e-9
assert abs(interaction_2way('B', 'C') - 0.5) < 1e-9

gap = {x: abs(add[x] - sub[x]) for x in COMPS}
flag = [x for x in COMPS if gap[x] > 0.3]
print(f'\n加法/减法差距 > 0.3 的组件：{[NAMES[x] for x in flag]}'
      f' -> **存在显著交互，需要单独研究**')
print(f'  差距最小的是 {NAMES[min(gap, key=gap.get)]}（{min(gap.values()):.2f}）'
      f' —— 它与别人的交互相互抵消了，两种读法一致')
assert flag == ['A', 'C'], flag
assert min(gap, key=gap.get) == 'B'
print('\n✅ 三条实用规则：')
print('   ① 决定"上不上线" -> 用**减法式**（部署形态就是完整系统）')
print('   ② 决定"下一步做什么" -> 用**加法式**（起点是当前系统）')
print('   ③ 组件数 k<=4 -> 直接跑全子集（2^4=16 组），顺带把交互作用也测出来')
print('   预算不够时的零成本退路：**同时报两端**，差距大的组件标注"存在交互"。')
print('\n📌 TSR 落点：Mosaic 与 copy-paste 都在制造更多小目标 —— 冗余是可预期的。')
print('    知道它们冗余，就可以砍掉一个省训练时间，而这个结论比"Mosaic 涨 1.0"有用得多。')

## 9 · 调参预算：「baseline 没调参」能偷走多少个点

只报最优 = 报 $n$ 次抽样的最大值，而最大值随 $n$ 单调增长，**与方法好坏无关**。
$$P(\text{至少一次超过 baseline}) = 1-(1-p)^n$$

In [ ]:
def p_at_least_one_beats(p, n):
    return 1 - (1 - p) ** n

print('单次抽样超过 baseline 的概率 p，与搜索 n 组后"至少找到一组更好"的概率：')
print(f"{'p':>6s}" + ''.join(f'{n:>9d}' for n in [1, 5, 10, 20, 50]))
for p in [0.05, 0.10, 0.20]:
    print(f'{p:>6.2f}' + ''.join(f'{p_at_least_one_beats(p, n):>9.1%}' for n in [1, 5, 10, 20, 50]))
assert abs(p_at_least_one_beats(0.10, 20) - 0.8784) < 1e-3
print('\n⚠️  **即使方法一点不比 baseline 好，只要允许随机调 20 组超参，')
print('    你有 87.8% 的概率能找到一组"涨了"的配置。** 这不是作弊，是搜索的数学性质。')

In [ ]:
# —— 预算-期望最优曲线：不用额外实验，用已跑过的结果重采样估计 ——
def expected_best_curve(results, budgets, trials=20000, seed=0):
    '''若只有预算 n 组，期望能拿到的最优值是多少（有放回重采样）。'''
    v = np.asarray(results, float)
    r = np.random.default_rng(seed)
    out = {}
    for n in budgets:
        idx = r.integers(0, len(v), size=(trials, n))
        out[n] = float(v[idx].max(1).mean())
    return out

# 方法 M：对超参敏感（方差大）；方法 R：鲁棒（方差小），二者的"极限最优"接近
rh = np.random.default_rng(0)
res_M = rh.normal(51.6, 0.90, size=200)     # 敏感：要搜很多组才能到高分
res_R = rh.normal(52.5, 0.25, size=200)     # 鲁棒：默认配置就不错
BUDGETS = [1, 2, 5, 10, 20, 50]
cM = expected_best_curve(res_M, BUDGETS, seed=1)
cR = expected_best_curve(res_R, BUDGETS, seed=2)

print(f"{'预算(组)':>9s} {'方法 M(敏感)':>14s} {'方法 R(鲁棒)':>14s} {'谁更好':>8s}")
for n in BUDGETS:
    print(f'{n:>9d} {cM[n]:>14.3f} {cR[n]:>14.3f} {("M" if cM[n] > cR[n] else "R"):>8s}')

assert cM[1] < cR[1], '小预算下鲁棒方法更好'
assert cM[50] > cM[1] + 1.5, '敏感方法的曲线爬升很多 —— 提升大半来自"搜索"而不是"方法"'
assert cR[50] - cR[1] < cM[50] - cM[1], '鲁棒方法的曲线更平'
print(f'\n方法 M 从预算 1 到 50：{cM[1]:.2f} -> {cM[50]:.2f}（爬升 {cM[50]-cM[1]:+.2f}）')
print(f'方法 R 从预算 1 到 50：{cR[1]:.2f} -> {cR[50]:.2f}（爬升 {cR[50]-cR[1]:+.2f}）')
print('\n⚠️  如果只报「最优值」：M 报 {:.2f}、R 报 {:.2f}，看起来 M 赢。'.format(cM[50], cR[50]))
print('    但在预算 1–5 组（= 真实项目的常态）时 **R 全面更好**，')
print('    而且 R 换数据集/换分辨率后不用重调 —— **"对超参不敏感"本身是重要优点**，')
print('    只报最优值的做法会把这个优点完全抹掉。')
print('\n✅ 面试答法：「我报的是等预算对比 —— 两个方法各搜 N 组，各自的最优。」')
print('   加分句：「而且我在两个模型尺寸上都验证了同一结论 —— ')
print('           如果提升只来自调参，它不会在两个尺寸上都成立。」')

## 10 · 实验记录 schema 与公平性审计器

两个可以直接搬进项目的小工具：
**① 记录校验**（缺字段就拒绝这次 run）、**② 公平性审计**（找出两个配置里"本不该不同"的项）。

In [ ]:
EXPERIMENT_SCHEMA = {
    'run_id':        ('str',  '唯一且人可读'),
    'baseline_run_id': ('str', '**对照是哪一次** —— 缺了它，这次实验没有可比对象'),
    'code_commit':   ('str',  '哪版代码'),
    'code_dirty':    ('bool', 'True = 这份代码在世界上任何地方都不存在'),
    'data_version':  ('str',  '含内容哈希，不是路径也不是日期'),
    'eval_version':  ('str',  '评测集版本（冻结）'),
    'eval_code_sha': ('str',  '评测代码版本 —— 评测代码改了，历史数字就不可比'),
    'config':        ('dict', '**完整展开**，不是与 base 的 diff'),
    'env':           ('dict', 'python/库/驱动/GPU'),
    'seed':          ('int',  '本模块全部内容的前提'),
    'metrics':       ('dict', '全量：整体 + 全部分桶 + 延迟 + 显存'),
    'wallclock_h':   ('num',  '没有它就算不出 ROI'),
}
TYPES = {'str': str, 'bool': bool, 'int': int, 'dict': dict, 'num': (int, float)}

def validate_record(rec, schema=EXPERIMENT_SCHEMA):
    problems = []
    for key, (typ, why) in schema.items():
        if key not in rec:
            problems.append(('missing', key, why))
        elif rec[key] is None:
            problems.append(('null', key, why))
        elif not isinstance(rec[key], TYPES[typ]) or (typ == 'int' and isinstance(rec[key], bool)):
            problems.append(('badtype', key, f'期望 {typ}，得到 {type(rec[key]).__name__}'))
    if rec.get('code_dirty') is True:
        problems.append(('blocker', 'code_dirty', '工作区脏 -> 不可复现，这次 run 不应产出正式结论'))
    mt = rec.get('metrics')
    if isinstance(mt, dict) and not any(k.startswith('by_') for k in mt):
        problems.append(('blocker', 'metrics', '只有整体指标、没有任何分桶 -> 无法归因'))
    return (not problems), problems

GOOD_REC = dict(
    run_id='2026-08-17_mosaic_s0', baseline_run_id='2026-08-17_base_s0',
    code_commit='d41d8cd98f00', code_dirty=False,
    data_version='tsr_v7.2#sha256:9f86d0', eval_version='tsr_eval_v3#frozen',
    eval_code_sha='a1b2c3d4',
    config={'lr': 0.01, 'epochs': 36, 'mosaic': True, 'close_mosaic': 10},
    env={'python': '3.11.9', 'numpy': '1.26.4', 'gpu': 'A100-80G'},
    seed=0,
    metrics={'mAP': 42.13, 'by_size': {'<16px': 21.4, '16-32': 38.7, '>32': 55.1},
             'by_light': {'day': 45.0, 'night': 33.2}, 'latency_p99_ms': 9.2},
    wallclock_h=11.4)
BAD_REC = json.loads(json.dumps(GOOD_REC))
BAD_REC['code_dirty'] = True
del BAD_REC['baseline_run_id']
BAD_REC['seed'] = None
BAD_REC['metrics'] = {'mAP': 42.13}

for label, rec in [('GOOD', GOOD_REC), ('BAD', BAD_REC)]:
    ok, probs = validate_record(rec)
    print(f'{label}: {"✅ 通过" if ok else "❌ 拒绝"}')
    for kind, key, why in probs:
        print(f'    [{kind:<8s}] {key:<16s} {why}')

assert validate_record(GOOD_REC)[0]
ok_b, probs_b = validate_record(BAD_REC)
assert not ok_b
kinds = {(k, key) for k, key, _ in probs_b}
assert ('missing', 'baseline_run_id') in kinds
assert ('null', 'seed') in kinds
assert ('blocker', 'code_dirty') in kinds
assert ('blocker', 'metrics') in kinds
print('\n✅ 四个问题各自对应一类无法回答的问题：')
print('   缺 baseline_run_id -> "这次和谁比的？" | seed=None -> "这是改动还是噪声？"')
print('   dirty=True -> "用的哪版代码？"        | 无分桶    -> "涨在哪？"')

In [ ]:
# —— 公平性审计：找出两个配置里"本不该不同"的项 ——
CONTROLLED = ['epochs', 'iterations', 'img_size', 'batch_size', 'lr', 'optimizer',
              'eval_version', 'eval_code_sha', 'score_thr', 'nms_iou', 'precision', 'seed_set']

def fairness_audit(cfg_a, cfg_b, tested_vars, controlled=CONTROLLED):
    '''tested_vars：本次实验**有意**改变的变量（含它的耦合组）。
       返回 (公平吗, 意外差异列表, 未记录的受控项列表)。'''
    diffs, missing = [], []
    for k in controlled:
        if k not in cfg_a or k not in cfg_b:
            missing.append(k)
            continue
        if cfg_a[k] != cfg_b[k] and k not in tested_vars:
            diffs.append((k, cfg_a[k], cfg_b[k]))
    return (not diffs and not missing), diffs, missing

BASE_CFG = dict(epochs=36, iterations=90000, img_size=640, batch_size=32, lr=0.01,
                optimizer='SGD', eval_version='v3', eval_code_sha='a1b2c3d4',
                score_thr=0.05, nms_iou=0.65, precision='fp16', seed_set='0,1,2',
                mosaic=False)
# 实验一：只改 mosaic —— 公平
EXP1 = dict(BASE_CFG, mosaic=True)
# 实验二：改了 mosaic，但顺手把 epochs 拉长、评测集也换了新版 —— 不公平
EXP2 = dict(BASE_CFG, mosaic=True, epochs=72, iterations=180000, eval_version='v4')

for label, cfg in [('实验一（只改 mosaic）', EXP1), ('实验二（顺手改了别的）', EXP2)]:
    fair, diffs, missing = fairness_audit(BASE_CFG, cfg, tested_vars={'mosaic'})
    print(f'{label}: {"✅ 公平" if fair else "❌ 不公平"}')
    for k, va, vb in diffs:
        print(f'    ⚠️ 意外差异  {k}: baseline={va}  实验={vb}')
    for k in missing:
        print(f'    ⚠️ 未记录     {k}')

assert fairness_audit(BASE_CFG, EXP1, {'mosaic'})[0]
fair2, diffs2, _ = fairness_audit(BASE_CFG, EXP2, {'mosaic'})
assert not fair2 and len(diffs2) == 3, diffs2
assert {k for k, _, _ in diffs2} == {'epochs', 'iterations', 'eval_version'}
# 未记录受控项也算不公平（因为你无法证明它相同）
_, _, miss3 = fairness_audit({'epochs': 36}, {'epochs': 36}, set())
assert len(miss3) == len(CONTROLLED) - 1
print('\n✅ 实验二的三项意外差异里，**换评测集是最致命的** ——')
print('   前两项让结论有偏，第三项让两个数字根本不可比。')
print('📌 把这个审计器接进实验提交流程：不公平的实验**不允许写进对比表**。')

## ✏️ 练习 1：方差预算与所需种子数

实现两个函数：
- `sd_of_mean_diff(sigma_s, sigma_e, n, paired)` → **两个配置各跑 n 个种子后，均值之差**的标准差
  （配对：$\sqrt2\,\sigma_e/\sqrt n$；非配对：$\sqrt2\sqrt{\sigma_s^2+\sigma_e^2}/\sqrt n$）
- `seeds_needed(delta, sigma_s, sigma_e, paired, alpha=0.05, power=0.8)`
  → 闭式所需的 n（向上取整）；用 $n \ge 7.849\,\sigma_\Delta^2/\delta^2$，其中 $\sigma_\Delta$ 是 **n=1** 时的差值标准差

In [ ]:
def sd_of_mean_diff(sigma_s, sigma_e, n, paired):
    # TODO
    raise NotImplementedError

def seeds_needed(delta, sigma_s, sigma_e, paired, alpha=0.05, power=0.8):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert abs(sd_of_mean_diff(0.22, 0.12, 1, True)  - SD_PAIRED)   < 1e-12
assert abs(sd_of_mean_diff(0.22, 0.12, 1, False) - SD_UNPAIRED) < 1e-12
assert abs(sd_of_mean_diff(0.22, 0.12, 4, True)  - SD_PAIRED / 2) < 1e-12   # 1/sqrt(4)
assert sd_of_mean_diff(0.22, 0.12, 5, True) < sd_of_mean_diff(0.22, 0.12, 5, False)
# sigma_s=0 时配对没有任何优势（种子效应本来就不存在）
assert abs(sd_of_mean_diff(0.0, 0.12, 3, True) - sd_of_mean_diff(0.0, 0.12, 3, False)) < 1e-12

assert seeds_needed(0.3, 0.22, 0.12, paired=True)  == 3
assert seeds_needed(0.3, 0.22, 0.12, paired=False) == 11
assert seeds_needed(1.0, 0.22, 0.12, paired=False) == 1
assert seeds_needed(0.1, 0.22, 0.12, paired=False) == 99
assert seeds_needed(0.3, 0.22, 0.12, True) < seeds_needed(0.3, 0.22, 0.12, False)

print(f"{'delta':>7s} {'配对 n':>8s} {'非配对 n':>10s} {'配对省下的训练次数':>20s}")
for d_ in [0.2, 0.3, 0.5, 1.0]:
    a_ = seeds_needed(d_, 0.22, 0.12, True); b_ = seeds_needed(d_, 0.22, 0.12, False)
    print(f'{d_:>7.1f} {a_:>8d} {b_:>10d} {2*(b_-a_):>20d}')
print('✅ 练习 1 通过：**配对是免费的方差削减** —— 同样的算力，能看见更小的效应。')

## ✏️ 练习 2：Holm–Bonferroni 逐步降级法

Bonferroni 太保守，BH 只控 FDR。**Holm** 在严格控制 FWER 的同时比 Bonferroni 更强。

实现 `holm(pvals, alpha=0.05)`：把 $p$ 升序排，依次检查 $p_{(k)} \le \alpha/(m-k+1)$，
**第一次失败就停止**，拒绝此前的全部。返回被拒绝的**原始下标**升序列表。

In [ ]:
def holm(pvals, alpha=0.05):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测（手算）——
P2 = [0.004, 0.006, 0.030]        # m=3，阈值依次 0.05/3=0.01667, 0.05/2=0.025, 0.05/1=0.05
assert holm(P2) == [0, 1, 2], holm(P2)             # 三个都通过
assert bonferroni(P2)[0] == [0, 1]                 # Bonferroni 只认前两个（阈值恒为 0.01667）
P3 = [0.004, 0.030, 0.006]                         # 打乱顺序，结果应与 P2 相同（按原始下标）
assert holm(P3) == [0, 1, 2], holm(P3)
assert holm([0.02, 0.03, 0.04]) == [], '第一步 0.02 > 0.05/3 就该全部停下'
assert holm(P, 0.05) == [0], holm(P, 0.05)         # 第 6 节那组 p 值
assert set(holm(P)) <= set(benjamini_hochberg(P)), 'Holm(控 FWER) 必然不比 BH(控 FDR) 激进'
print(f'P2 = {P2}')
print(f'  Bonferroni -> {bonferroni(P2)[0]}   Holm -> {holm(P2)}   （Holm 严格更强，且仍控 FWER）')
print(f'P  = {P}')
print(f'  Bonferroni -> {bonferroni(P)[0]}   Holm -> {holm(P)}   BH -> {benjamini_hochberg(P)}')
print('✅ 练习 2 通过：门禁用 Holm（控 FWER 但比 Bonferroni 少漏），探索用 BH。')

## ✏️ 练习 3：通用 Shapley 归因

实现 `shapley_values(perf_fn, components)`：对任意 `perf_fn(子集) -> 数值` 与组件列表，
返回 `{组件: Shapley 值}`。

$$\phi_i=\sum_{S\subseteq N\setminus\{i\}}\frac{|S|!\,(n-|S|-1)!}{n!}\bigl[f(S\cup\{i\})-f(S)\bigr]$$

In [ ]:
def shapley_values(perf_fn, components):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
sv = shapley_values(perf, COMPS)
assert abs(sv['A'] - 0.5166666667) < 1e-6, sv
assert abs(sv['B'] - 0.9166666667) < 1e-6, sv
assert abs(sv['C'] - 0.7666666667) < 1e-6, sv
# **效率公理**：分摊之和 == 总增益
assert abs(sum(sv.values()) - (perf(COMPS) - perf([]))) < 1e-9

# 无交互时，三种归因必须完全一致（这是最好的正确性检查）
def perf_additive(S):
    return 40.0 + sum({'X': 1.0, 'Y': 0.5, 'Z': 0.2}[c] for c in S)
sv2 = shapley_values(perf_additive, ['X', 'Y', 'Z'])
assert abs(sv2['X'] - 1.0) < 1e-12 and abs(sv2['Y'] - 0.5) < 1e-12 and abs(sv2['Z'] - 0.2) < 1e-12

# **虚拟成员公理**：不贡献任何东西的组件，Shapley 值必须是 0
def perf_dummy(S):
    return perf_additive([c for c in S if c != 'W'])
sv3 = shapley_values(perf_dummy, ['X', 'Y', 'Z', 'W'])
assert abs(sv3['W']) < 1e-12, sv3

print('含交互:', {k: round(v, 4) for k, v in sv.items()},  '合计', round(sum(sv.values()), 4))
print('无交互:', {k: round(v, 4) for k, v in sv2.items()}, '-> 与加法式/减法式完全一致')
print('虚拟项:', {k: round(v, 4) for k, v in sv3.items()}, '-> W 的贡献恰为 0')
print('✅ 练习 3 通过：**无交互时三种归因一致 —— 所以它们的分歧本身就是交互的度量。**')

## ✏️ 练习 4：预算-期望最优曲线的闭式解

不用重采样也能算。把 $N$ 个结果升序排成 $v_{(1)}\le\dots\le v_{(N)}$，
有放回抽 $n$ 次的最大值期望是：

$$\mathbb E[\max] = \sum_{i=1}^{N} v_{(i)}\left[\left(\tfrac iN\right)^{n}-\left(\tfrac{i-1}{N}\right)^{n}\right]$$

实现 `expected_best_exact(results, n)`，并与第 9 节的蒙特卡洛版本对拍。

In [ ]:
def expected_best_exact(results, n):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测（手算）——
assert abs(expected_best_exact([1, 2, 3, 4], 1) - 2.5) < 1e-12
# n=2: (1*1 + 2*3 + 3*5 + 4*7)/16 = 50/16 = 3.125
assert abs(expected_best_exact([1, 2, 3, 4], 2) - 3.125) < 1e-12
assert abs(expected_best_exact([5.0], 7) - 5.0) < 1e-12
assert expected_best_exact([1, 2, 3, 4], 10) > expected_best_exact([1, 2, 3, 4], 3)

# 与蒙特卡洛对拍（这是验证闭式实现最可靠的方式）
print(f"{'预算':>6s} {'蒙特卡洛':>10s} {'闭式':>10s} {'差':>9s}")
for n_ in BUDGETS:
    mc_ = cR[n_]; ex_ = expected_best_exact(res_R, n_)
    print(f'{n_:>6d} {mc_:>10.4f} {ex_:>10.4f} {abs(mc_-ex_):>9.5f}')
    assert abs(mc_ - ex_) < 0.02, (n_, mc_, ex_)
print('✅ 练习 4 通过：**已经跑过的实验就够画出预算曲线了，不需要新实验。**')
print('   把它加进你的实验报告，「调参不公平」这个质疑就自动被回答了。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def sd_of_mean_diff(sigma_s, sigma_e, n, paired):
    per_run_var = sigma_e ** 2 if paired else sigma_s ** 2 + sigma_e ** 2
    return math.sqrt(2 * per_run_var / n)

def seeds_needed(delta, sigma_s, sigma_e, paired, alpha=0.05, power=0.8):
    z = ND.inv_cdf(1 - alpha / 2) + ND.inv_cdf(power)
    sd1 = sd_of_mean_diff(sigma_s, sigma_e, 1, paired)      # n=1 时的差值标准差
    return math.ceil(z * z * sd1 * sd1 / (delta * delta))

In [ ]:
# 练习 2 参考答案
def holm(pvals, alpha=0.05):
    m = len(pvals)
    order = sorted(range(m), key=lambda i: pvals[i])
    rejected = []
    for k, i in enumerate(order, start=1):
        if pvals[i] <= alpha / (m - k + 1):
            rejected.append(i)
        else:
            break                                   # **第一次失败就停止**
    return sorted(rejected)

In [ ]:
# 练习 3 参考答案
def shapley_values(perf_fn, components):
    comps = list(components)
    n = len(comps)
    out = {}
    for x in comps:
        others = [c for c in comps if c != x]
        tot = 0.0
        for r in range(len(others) + 1):
            for S in itertools.combinations(others, r):
                w = math.factorial(len(S)) * math.factorial(n - len(S) - 1) / math.factorial(n)
                tot += w * (perf_fn(list(S) + [x]) - perf_fn(list(S)))
        out[x] = tot
    return out

In [ ]:
# 练习 4 参考答案
def expected_best_exact(results, n):
    v = np.sort(np.asarray(results, float))
    N = len(v)
    i = np.arange(1, N + 1)
    w = (i / N) ** n - ((i - 1) / N) ** n           # P(max 恰好是第 i 小的那个)
    return float((v * w).sum())

---
## 🧪 真实工程胶囊：一次可信实验的完整流程

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════════
# 「我要验证 X 有没有用」—— 从提出假设到写下结论的完整流程
# 每一步都有明确的产出物；缺任何一步，结论的强度都要打折并**如实标注**
# ══════════════════════════════════════════════════════════════════════

# ─── 阶段 0：开跑之前（最重要，也最常被跳过）────────────────────────
# 0.1 写下**事前预测**（预注册）。这一条堵死了"事后找解释"这个最强的自欺来源
#     assumption.md:
#       假设：copy-paste 增强能提升稀有类召回
#       机制：稀有类样本量从 30 -> 300，正样本数是当前瓶颈
#       预测：尾部类 AP +3~8；整体 mAP +0.2~0.6；head 类不动（±0.2 内）
#       延迟：0（训练期改动）      显存：+0（离线合成）
#       **若整体涨了但尾部类没涨，说明机制判断错了，不能算验证成功**
#
# 0.2 算 MDE：我这次预算能看见多大的效应？
#     sigma = 项目 baseline 的多种子标准差（模块 00 第一周检查单第③项）
#     MDE_closed = 2.80 * sqrt(2)*sigma_e / sqrt(n)   # 记得：**闭式会乐观一倍**
#     若 MDE > 预测效应 -> **这个实验不值得跑**，先加预算或改设计
#
# 0.3 公平性自查（fairness_audit）：列出本次**有意**改变的变量及其耦合组，
#     其余受控项逐项对齐。特别注意：epoch/iteration、增强、分辨率、
#     score/NMS 阈值、评测集版本、评测代码版本、精度、硬件

# ─── 阶段 1：跑 ────────────────────────────────────────────────────
# 1.1 **配对**：两个配置用同一组种子
#     for s in 0 1 2 3 4; do
#       train.py --cfg base.yaml      --seed $s --tag base_s$s
#       train.py --cfg base+cp.yaml   --seed $s --tag cp_s$s
#     done
# 1.2 baseline **必须在同一个 commit 上重跑**，不许用历史数字
# 1.3 每个 run 落 manifest（validate_record 不过就不产出）

# ─── 阶段 2：分析 ──────────────────────────────────────────────────
# 2.1 配对诊断：pairing_diagnostic(A, B) —— 削减 < 20% 说明配对失效，要加种子
# 2.2 配对 t 检验 + 95% CI；**报效应量与 CI，p 值只是附带**
# 2.3 分桶逐个看：涨的是不是**预测的那个桶**？其他桶有没有掉？
# 2.4 多重比较：本季度做了几次对比？超过 5 次就上 Holm 或 BH

# ─── 阶段 3：写结论（模板，直接套用）──────────────────────────────
#   【结论】在 <同 commit / 36 epoch / 640 输入 / 同增强其余项 / eval_v3 冻结> 下，
#           copy-paste 使 **尾部类 AP +4.2（95% CI [2.1, 6.3]，n=5 配对，p=0.004）**，
#           整体 mAP +0.31（95% CI [0.05, 0.57]），head 类 -0.08（在噪声内）。
#   【代价】训练时长 +8%，显存不变，推理延迟不变；实例库构建一次性 6 人时。
#   【局限】只在 tsr_v7.2 与 640 分辨率下验证；1280 下未测；
#           尾部类样本量 <10 的极稀有类仍无提升（n 太小，本次 MDE 覆盖不到）。
#   【下一步】按 0.1 的机制推断，应对 <10 张的极稀有类改用两级架构（C55 m02）。
#
#   ▶ **四段缺一不可**：只有【结论】没有【代价】和【局限】的报告，
#     在面试和在评审会上会被同样的追问打穿。

# ─── 反模式速查 ───────────────────────────────────────────────────
#  · "涨了 0.4，上了"                 -> 没有 sigma，不知道 0.4 是不是噪声
#  · "baseline 用的论文数字"           -> 基线漂移，对比无效
#  · "顺手把 epoch 也拉长了"           -> 混杂因子，归因失效
#  · "这个 p=0.03，所以有效"           -> 只报 p 不报效应量与 CI
#  · "试了 20 个改动，这个显著"        -> 多重比较，FWER 已达 64%
#  · "新方法搜了 200 组，baseline 默认" -> 调参不公平，报预算曲线
'''
print(RECIPE)
for token in ['事前预测', 'MDE', 'fairness_audit', '配对', 'pairing_diagnostic',
              '95% CI', '分桶', '多重比较', '【代价】', '【局限】', '反模式']:
    assert token in RECIPE, token
print('✅ 流程覆盖：预注册 -> 功效 -> 公平性 -> 配对执行 -> 统计分析 -> 四段式结论')

### 小结

- **种子方差是这门课的第一性事实**：$\sigma\approx0.25$ 时，5 个种子的期望极差是 **0.58 个 mAP**，
  而它们是完全相同的配置。**真实差异为 0 时，单次非配对对比有 19.9% 的概率显示「+0.3 的提升」。**
- **「+0.3 算提升吗」的满分答法**：先反问口径（哪个指标 / 几个种子 / 配对没有）→ 给出 20% 这个数
  → 说该怎么做（配对 3–5 组 + 分桶一致性）→ 给出退路（算力不够时用多个弱证据的一致性）。
- **配对是免费的方差削减**：同一组种子跑两个配置，种子效应在做差时抵消。
  本模块的例子里，**同一批数据配对分析 $p=0.002$、非配对分析 $p=0.20$** ——
  差别完全来自实验设计。但要跑 `pairing_diagnostic` 确认它真的生效了。
- **功效分析要在开跑前做**。检出 +0.3 需要非配对 22 次训练 / 配对 6 次（闭式）；
  而**闭式用正态近似，小样本下会低估**——蒙特卡洛给出配对实际需要 5 对。
  预算只有 3 对时，真实 MDE 是 **0.56 而不是 0.27**。
  MDE 之下只能报「没有证据说明它有用」，不能报「有证据说明它没用」。
- **多重比较**：20 次对比 → 至少一个假阳性的概率 **64%**。
  门禁用 Bonferroni/Holm（控 FWER），探索用 BH（控 FDR）；比统计校正更好用的是
  **dev/holdout 分阶段验证**。
- **消融的加法式与减法式会给出相反结论**：例子里 A 的贡献是 +1.00 还是 +0.10 相差 10 倍，
  而两者都没算错。**它们的分歧本身就是交互作用的度量**（无交互时三种归因完全一致）。
  $k\le4$ 时直接跑全子集（16 组），顺带把交互测出来。
- **只报最优值等于报 $n$ 次抽样的最大值**：即使方法一点不比 baseline 好，
  随机搜 20 组也有 **87.8%** 的概率找到「涨了」的配置。用**预算-期望最优曲线**
  （已跑过的实验就够算，还有闭式解）来做等预算对比。
- **结论必须四段式**：结论（含全部实验条件）/ 代价 / 局限 / 下一步。
  少了【代价】与【局限】的报告，在评审会和面试里会被同样的追问打穿。

下一站：**模块 02 · 误差分析工程** —— 实验证明了「有用」之后，
下一个问题是「现在该做什么」：把 mAP 的损失拆成 Cls/Loc/Dupe/Bkg/Miss 六类，
并算出**修好每一类能涨多少**。